In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_csv("../../data/Customer-Churn-Records.csv")

In [ ]:
df.shape

In [ ]:
df.head()

In [ ]:
df.info()

No missing data — all columns are non-null.

In [ ]:
df.describe()

In [ ]:
(df["Complain"] == df["Exited"]).sum()

## ⚠️ Finding: Data Leakage - Complain column

For 9,986 out of 10,000 customers, the `Complain` column has exactly the same value as `Exited`.
This means that almost every customer who left had previously filed a complaint.
This column cannot be used in the model, because in real life the bank will not know
in advance whether a customer will file a complaint. This is called **data leakage**.

➡️ Passing this to Michał — the Complain column should be removed from the model.

In [ ]:
df["Exited"].value_counts()

## 2,038 customers left — 20.38% churn rate

Every 5th bank customer closed their account — meaning we lose 1 in 5 customers.

In [ ]:
sns.countplot(data=df, x="Exited", legend='auto')
plt.title("Customer Churn Distribution")
plt.savefig("../../reports/figures/Customer_Churn_Distribution.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
sns.histplot(data=df, x="Age", hue="Exited", kde=True)
plt.title("Age Distribution by Churn Status")
plt.savefig("../../reports/figures/Age_Distribution.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle("Distribution of Numerical Features by Churn Status", y=0.93, fontsize=16)
sns.histplot(data=df, x="Balance", hue="Exited", ax=axes[0][0], kde=True)
sns.histplot(data=df, x="CreditScore", hue="Exited", ax=axes[0][1], kde=True)
sns.histplot(data=df, x="EstimatedSalary", hue="Exited", ax=axes[0][2])
sns.histplot(data=df, x="Tenure", hue="Exited", ax=axes[1][0])
sns.histplot(data=df, x="NumOfProducts", hue="Exited", ax=axes[1][1])
sns.histplot(data=df, x="Point Earned", hue="Exited", ax=axes[1][2])
plt.savefig("../../reports/figures/Numerical_Features_Distribution.png", dpi=150, bbox_inches="tight")
plt.show()

1. Looking at the Balance charts, the large bar at 0 represents customers with zero balance. 
customers with zero balance churn less frequently — the orange bar at 0 is very small 
compared to the blue one.

2. CreditScore distribution is similar for both groups — no clear difference is visible.

3. No conclusions can be drawn from EstimatedSalary — evenly distributed across both groups.

4. At Tenure=0 (less than a year with the bank), the orange bar is slightly larger 
proportionally. Overall, no significant difference across the remaining values.

5. Customers with 3 or 4 NumOfProducts churn more frequently than those with 1 or 2 — 
the bank should investigate whether conditions for multi-product customers are sufficiently attractive.

6. No conclusions can be drawn from Point Earned — distribution is similar for both groups.

In [ ]:
fig1, axes1 = plt.subplots(2, 3, figsize=(15, 10))
axes1[1][2].set_visible(False)
fig1.suptitle("Distribution of Categorical Features by Churn Status", y=0.93, fontsize=16)
sns.countplot(data=df, x="Geography", hue="Exited", ax=axes1[0][0])
sns.countplot(data=df, x="Gender", hue="Exited", ax=axes1[0][1])
sns.countplot(data=df, x="Card Type", hue="Exited", ax=axes1[0][2])
sns.countplot(data=df, x="HasCrCard", hue="Exited", ax=axes1[1][0])
sns.countplot(data=df, x="IsActiveMember", hue="Exited", ax=axes1[1][1])
plt.savefig("../../reports/figures/Categorical_Features_Distribution.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
geography_mean = df.groupby(["Geography"])["Exited"].mean()
geography_mean

In [ ]:
gender_mean = df.groupby(["Gender"])["Exited"].mean()
gender_mean

In [ ]:
card_type_mean = df.groupby(["Card Type"])["Exited"].mean()
card_type_mean

In [ ]:
has_cr_card_mean = df.groupby(["HasCrCard"])["Exited"].mean()
has_cr_card_mean

In [ ]:
is_active_member_mean = df.groupby(["IsActiveMember"])["Exited"].mean()
is_active_member_mean

1. Customers in Germany churn twice as often as in France or Spain — 32% vs 16%.

2. Female customers churn more frequently than male customers — 25% vs 16%.

3. Card Type has no impact on churn — all card types show a similar churn rate (~20%).

4. Having a credit card or not has almost no impact on churn — 20.2% vs 20.8%, 
a negligible difference.

5. Inactive members churn almost twice as often as active ones — 26.8% vs 14.2%.

In [ ]:
plt.figure(figsize=(10, 8))
cols_to_drop = ["RowNumber", "CustomerId"]
sns.heatmap(df.select_dtypes(include="number").drop(columns=cols_to_drop).corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Heatmap")
plt.savefig("../../reports/figures/Correlation_Heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

Summarizing the correlations with Exited:

* Exited and Complain have a correlation of 1.00 — confirming the data leakage finding,
* Age: 0.29,
* Balance: 0.12,
* IsActiveMember: -0.16 (negative correlation — the less active the customer, 
the more likely they are to churn)

Everything aligns with what we observed earlier in the countplots.